# Supervised Genre Classification

**Pipeline Overview:**
1. Replicate Weighted TF-IDF + 30-component Truncated SVD from the exploratory notebook
2. Combine latent textual features with `rating_count`, `review_count`, `avg_rating`
3. Extract dominant genre (first listed) and filter to top 10 by frequency
4. Train Random Forest, Gradient Boosting, and Logistic Regression
5. Evaluate with Stratified 5-Fold CV (Accuracy + Macro F1)
6. Export predictions from the best model to `predictions.csv`

## 1 — Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import lil_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time

# ── Reproducibility ──
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plot styling (matches v4.ipynb) ──
sns.set_style("whitegrid")
DARK = "#0E1117"
plt.rcParams.update({
    "figure.dpi": 150, "font.family": "sans-serif",
    "axes.titleweight": "bold", "axes.labelweight": "bold",
    "figure.facecolor": DARK, "axes.facecolor": DARK,
    "text.color": "#E0E0E0", "axes.labelcolor": "#E0E0E0",
    "xtick.color": "#AAAAAA", "ytick.color": "#AAAAAA",
    "grid.color": "#2A2D35", "grid.alpha": 0.5,
})
ACCENT = ["#FF6B6B", "#4ECDC4", "#FFE66D", "#A06CD5", "#FF9F43",
          "#54A0FF", "#5CD85C", "#FD7272", "#C56CF0", "#17C0EB"]

print("All libraries imported successfully.")

## 2 — Data Ingestion & Cleaning
Exact replication of the `load_data()` function from `v4.ipynb`, including descriptor
cleaning, noise-tag filtering, and date parsing.

In [ ]:
# ── Noise tags to exclude (identical to v4.ipynb) ──
NOISE_TAGS = frozenset({
    "malevocals", "femalevocals", "maleandfemalevocals",
    "instrumental", "vocals", "vocal", "conceptalbum", "concept",
    "album", "acoustic", "electric", "production", "lo-fiproduction",
    "liveperformance", "live", "liverecording", "studio", "studioalbum",
    "remaster", "remastered", "reissue", "compilation", "soundtrack",
    "ost", "debut", "debutalbum", "sideproject", "collaboration",
    "ep", "single", "mixtape", "bonus", "bonustracks", "deluxe",
    "explicit", "clean", "mono", "stereo",
    "avant-garde", "choral", "improvisation", "monologue",
    "opera", "orchestral", "rockopera", "suite", "carol",
    "ensemble", "vocalgroup", "wallofsound", "androgynousvocals",
    "lo-fi", "symphony", "jingle", "chambermusic", "generativemusic",
    "sonata", "acappella", "...",
    "melodic", "energetic"
})


def load_data(path: str) -> pd.DataFrame:
    """
    Load and clean the dataset — exact replica of v4.ipynb logic.
    Steps:
      1. Parse comma-separated descriptors into cleaned tag lists.
      2. Filter out noise tags and rows with no valid tags.
      3. Parse release dates; drop rows with invalid dates.
      4. Select the working column set.
    """
    df = pd.read_csv(path, encoding="latin-1")

    def clean_descriptors(raw):
        if pd.isna(raw):
            return []
        tags = [t.strip().lower().replace(" ", "") for t in str(raw).split(",")]
        return [t for t in tags if t and len(t) > 1 and t not in NOISE_TAGS]

    df["tags"] = df["descriptors"].apply(clean_descriptors)
    df = df[df["tags"].apply(len) > 0].reset_index(drop=True)

    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
    df = df.dropna(subset=["release_date"]).reset_index(drop=True)
    df["date_obj"] = df["release_date"].dt.to_period("M").dt.to_timestamp()
    df["year"] = df["date_obj"].dt.year.astype(int)

    df = df[['position', 'release_name', 'artist_name', 'primary_genres',
             'secondary_genres', 'avg_rating', 'rating_count', 'review_count',
             'year', 'date_obj', 'tags']]
    return df


# ── Load the dataset ──
DATA_PATH = "rym_clean1.csv"  # Update path as needed
df = load_data(DATA_PATH)

print(f"Dataset loaded: {df.shape[0]} albums × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head(3)

## 3 — Feature Engineering: Positional-Weighted TF-IDF + Truncated SVD
Re-implements the positional decay TF-IDF weighting scheme (`α=0.50`) and
reduces the sparse tag matrix to **30 dense SVD components** as specified.
These are then combined with the three numerical features.

In [ ]:
def positional_decay_weights(n: int, alpha: float = 0.50) -> np.ndarray:
    """
    Exponential positional decay: w[0]=1.0, w[n-1]≈alpha.
    Identical to v4.ipynb implementation.
    """
    if n <= 1:
        return np.ones(n, dtype=np.float64)
    lam = -np.log(alpha) / (n - 1)
    return np.exp(-lam * np.arange(n))


def build_weighted_tfidf(df: pd.DataFrame):
    """
    Construct a positionally-weighted TF-IDF sparse matrix.
    Returns: M_norm (csr_matrix), vocab (list), tag2col (dict), idf (ndarray)
    """
    # --- Document frequency per tag ---
    tag_sets = df["tags"].apply(set)
    vocab_counter = {}
    for ts in tag_sets:
        for t in ts:
            vocab_counter[t] = vocab_counter.get(t, 0) + 1

    vocab = sorted(vocab_counter.keys())
    tag2col = {t: i for i, t in enumerate(vocab)}
    n_albums = len(df)
    n_vocab = len(vocab)

    # --- Smooth IDF: log((1+N)/(1+df_t)) + 1 ---
    df_arr = np.array([vocab_counter[t] for t in vocab], dtype=np.float64)
    idf = np.log((1.0 + n_albums) / (1.0 + df_arr)) + 1.0

    # --- Populate sparse matrix with positional_weight × IDF ---
    M = lil_matrix((n_albums, n_vocab), dtype=np.float64)
    for row_idx, tags in enumerate(df["tags"]):
        n = len(tags)
        weights = positional_decay_weights(n, alpha=0.50)
        for pos, tag in enumerate(tags):
            if tag in tag2col:
                col = tag2col[tag]
                M[row_idx, col] = weights[pos] * idf[col]

    M_csr = M.tocsr()
    M_norm = normalize(M_csr, norm="l2", axis=1)  # L2-normalize rows

    sparsity = 1.0 - M_csr.nnz / (n_albums * n_vocab)
    print(f"  TF-IDF Matrix: {n_albums} albums × {n_vocab} tags")
    print(f"  IDF range: [{idf.min():.3f}, {idf.max():.3f}]")
    print(f"  Sparsity: {sparsity:.2%}")

    return M_norm, vocab, tag2col, idf


# ── Build TF-IDF ──
print("Building positionally-weighted TF-IDF matrix...")
M, vocab, tag2col, idf = build_weighted_tfidf(df)

# ── Truncated SVD: 30 Components ──
N_COMPONENTS = 30
print(f"\nFitting Truncated SVD with k={N_COMPONENTS} components...")
svd_model = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE,
                          algorithm="randomized")
Z = svd_model.fit_transform(M)

explained_var = svd_model.explained_variance_ratio_.sum()
print(f"  Cumulative explained variance at k={N_COMPONENTS}: {explained_var:.2%}")
print(f"  Latent matrix Z shape: {Z.shape}")

## 4 — Construct the Final Feature Matrix
Combine the 30 SVD latent dimensions with `rating_count`, `review_count`, and
`avg_rating` to form a **33-feature** independent variable matrix.

In [ ]:
# ── 30 SVD features as a DataFrame ──
svd_feature_names = [f"svd_{i}" for i in range(N_COMPONENTS)]
df_svd = pd.DataFrame(Z, columns=svd_feature_names, index=df.index)


# ── Concatenate: 30 + 3 = 33 features ──
X_combined = pd.concat([df_svd], axis=1)

print(f"Feature matrix constructed: {X_combined.shape}")
print(f"  - SVD latent features: {len(svd_feature_names)}")
print(f"  - Numerical features:  0")
print(f"  - Total features:      {X_combined.shape[1]}")
X_combined.head(3)

## 5 — Target Variable Preprocessing
Extract the **first genre** from the comma-separated `primary_genres` column as the
dominant genre. Then filter to the **top 10 most frequent** dominant genres to ensure
sufficient sample sizes per class for robust classification.

In [ ]:
# ── Step 1: Extract dominant genre (first before comma) ──
df["dominant_genre"] = df["primary_genres"].apply(
    lambda x: str(x).split(",")[0].strip() if pd.notna(x) else "Unknown"
)

# ── Step 2: Identify top 10 genres by frequency ──
genre_counts = df["dominant_genre"].value_counts()
top_10_genres = genre_counts.head(10).index.tolist()

print("Top 10 Dominant Genres by Frequency:")
for i, genre in enumerate(top_10_genres, 1):
    print(f"  {i:>2}. {genre:<30s}  {genre_counts[genre]:>4d} albums")
print(f"\n  Total albums in top 10: {genre_counts[top_10_genres].sum()}")
print(f"  Coverage: {genre_counts[top_10_genres].sum() / len(df):.1%} of dataset")

# ── Step 3: Filter to top 10 genres ──
mask_top10 = df["dominant_genre"].isin(top_10_genres)
X_filtered = X_combined.loc[mask_top10].reset_index(drop=True)
y_filtered = df.loc[mask_top10, "dominant_genre"].reset_index(drop=True)

# Preserve original indices for the final output DataFrame
original_indices = df.loc[mask_top10].index.tolist()

# ── Encode string labels to integers ──
le = LabelEncoder()
y_encoded = le.fit_transform(y_filtered)

print(f"\nFiltered dataset: {X_filtered.shape[0]} samples × {X_filtered.shape[1]} features")
print(f"Number of classes: {len(le.classes_)}")
print(f"Classes: {list(le.classes_)}")

### Class Distribution Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(DARK)

genre_dist = y_filtered.value_counts()
bars = ax.barh(genre_dist.index[::-1], genre_dist.values[::-1],
               color=ACCENT[:len(genre_dist)], edgecolor="none", height=0.7)

# Add count labels on each bar
for bar, count in zip(bars, genre_dist.values[::-1]):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
            f"{count}", va="center", ha="left", fontsize=9, color="#E0E0E0")

ax.set_xlabel("Number of Albums")
ax.set_title("Class Distribution — Top 10 Dominant Genres",
             fontsize=13, pad=12, color="white")
ax.set_xlim(0, genre_dist.max() * 1.15)

plt.tight_layout()
plt.savefig("fig_class_distribution.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## 6 — Model Initialization
Three classifiers are instantiated with hyperparameters tuned for this task:
- **Random Forest** — ensemble of 300 fully-grown trees with `class_weight="balanced"`
- **Gradient Boosting** — 200 boosted stumps (`max_depth=5`) with 0.8 subsampling
- **Logistic Regression** — multinomial softmax with `max_iter=2000` for convergence

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,           # Let trees grow fully
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",  # Address class imbalance
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=2,
        subsample=0.8,
        random_state=RANDOM_STATE
    ),
    "Logistic Regression": LogisticRegression(
        solver="lbfgs",             # Supports native multinomial
        max_iter=2000,              # Increased for convergence
        class_weight="balanced",
        C=1.0,
        random_state=RANDOM_STATE
    )
}

## 7 — Stratified 5-Fold Cross-Validation
Mandatory stratified splitting preserves class proportions in each fold.
Feature scaling is performed **within each fold** (fit on train, transform both)
to prevent data leakage. Tree-based models use raw features; Logistic Regression
uses standardized features.

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# ── Storage for per-fold metrics ──
results = {name: {"accuracy": [], "macro_f1": []} for name in models}

print(f"{'='*70}")
print(f" STRATIFIED {N_FOLDS}-FOLD CROSS-VALIDATION")
print(f"{'='*70}")

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_filtered, y_encoded), 1):
    print(f"\n─── Fold {fold_idx}/{N_FOLDS} ───  "
          f"(Train: {len(train_idx):,} | Val: {len(val_idx):,})")

    # ── Split data ──
    X_train_raw, X_val_raw = X_filtered.iloc[train_idx], X_filtered.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    # ── Per-fold scaling (fit on train only → no leakage) ──
    fold_scaler = StandardScaler()
    X_train_scaled = fold_scaler.fit_transform(X_train_raw)
    X_val_scaled = fold_scaler.transform(X_val_raw)

    for model_name, model in models.items():
        t0 = time.time()

        # Use scaled data only for Logistic Regression
        if model_name == "Logistic Regression":
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_val_scaled)
        else:
            model.fit(X_train_raw.values, y_train)
            y_pred = model.predict(X_val_raw.values)

        # ── Compute fold metrics ──
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average="macro")
        elapsed = time.time() - t0

        results[model_name]["accuracy"].append(acc)
        results[model_name]["macro_f1"].append(f1)

        print(f"  {model_name:<25s} → Acc: {acc:.4f}  |  "
              f"Macro-F1: {f1:.4f}  |  {elapsed:.1f}s")

## 8 — Cross-Validation Summary & Best Model Selection
Aggregate per-fold metrics into mean ± std and select the optimal model
by highest **Mean Macro-F1** (the primary metric for imbalanced multi-class tasks).

In [ ]:
print(f"{'='*70}")
print(f" CROSS-VALIDATION SUMMARY — Mean ± Std across {N_FOLDS} Folds")
print(f"{'='*70}\n")

summary_rows = []
for model_name in models:
    acc_arr = np.array(results[model_name]["accuracy"])
    f1_arr = np.array(results[model_name]["macro_f1"])
    summary_rows.append({
        "Model": model_name,
        "Mean Accuracy": acc_arr.mean(),
        "Std Accuracy": acc_arr.std(),
        "Mean Macro-F1": f1_arr.mean(),
        "Std Macro-F1": f1_arr.std()
    })
    print(f"  {model_name:<25s}")
    print(f"    Accuracy : {acc_arr.mean():.4f} ± {acc_arr.std():.4f}  "
          f"(per fold: {', '.join(f'{v:.4f}' for v in acc_arr)})")
    print(f"    Macro-F1 : {f1_arr.mean():.4f} ± {f1_arr.std():.4f}  "
          f"(per fold: {', '.join(f'{v:.4f}' for v in f1_arr)})")
    print()

summary_df = pd.DataFrame(summary_rows).sort_values("Mean Macro-F1", ascending=False)

# ── Determine the best model (highest Mean Macro-F1) ──
best_model_name = summary_df.iloc[0]["Model"]
best_f1 = summary_df.iloc[0]["Mean Macro-F1"]
best_acc = summary_df.iloc[0]["Mean Accuracy"]

print(f"  ★ Best Model: {best_model_name}")
print(f"    Mean Accuracy: {best_acc:.4f}")
print(f"    Mean Macro-F1: {best_f1:.4f}")

summary_df

### Cross-Validation Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(DARK)

metric_names = ["accuracy", "macro_f1"]
metric_labels = ["Accuracy", "Macro F1-Score"]

for ax, metric, label in zip(axes, metric_names, metric_labels):
    model_names_list = list(models.keys())
    means = [np.mean(results[m][metric]) for m in model_names_list]
    stds = [np.std(results[m][metric]) for m in model_names_list]

    bars = ax.bar(model_names_list, means, yerr=stds, capsize=5,
                  color=ACCENT[:3], edgecolor="none", alpha=0.85,
                  error_kw={"ecolor": "#AAAAAA", "capthick": 1.5})

    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + std + 0.005,
                f"{mean:.3f}", ha="center", va="bottom",
                fontsize=10, color="#E0E0E0", fontweight="bold")

    ax.set_ylabel(label)
    ax.set_title(f"CV {label} (Mean ± Std)", fontsize=12, pad=10, color="white")
    ax.set_ylim(0, max(means) * 1.20)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig("fig_cv_results.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## 9 — Fit Best Model on Full Filtered Dataset
Re-instantiate a fresh copy of the best-performing model (to avoid any state
from CV folds) and train it on **all** 1,323 filtered samples to generate
the final predictions for the deliverable.

In [ ]:
print(f"Fitting best model ({best_model_name}) on the FULL filtered dataset...")
print(f"  Training samples: {X_filtered.shape[0]}")
print(f"  Features: {X_filtered.shape[1]}")

# ── Fresh model instantiation (avoids leakage from CV state) ──
best_model_fresh = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_split=5,
        min_samples_leaf=2, max_features="sqrt", class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        min_samples_split=5, min_samples_leaf=2, subsample=0.8,
        random_state=RANDOM_STATE
    ),
    "Logistic Regression": LogisticRegression(
        solver="lbfgs", max_iter=2000,
        class_weight="balanced", C=1.0, random_state=RANDOM_STATE
    )
}[best_model_name]

# ── Fit on entire filtered dataset ──
if best_model_name == "Logistic Regression":
    scaler_final = StandardScaler()
    X_final_scaled = scaler_final.fit_transform(X_filtered)
    best_model_fresh.fit(X_final_scaled, y_encoded)
    y_pred_full = best_model_fresh.predict(X_final_scaled)
else:
    best_model_fresh.fit(X_filtered.values, y_encoded)
    y_pred_full = best_model_fresh.predict(X_filtered.values)

# ── Decode predictions back to genre strings ──
y_pred_labels = le.inverse_transform(y_pred_full)
y_actual_labels = le.inverse_transform(y_encoded)

# Resubstitution metrics (training set — for reference, CV is the real estimate)
full_acc = accuracy_score(y_encoded, y_pred_full)
full_f1 = f1_score(y_encoded, y_pred_full, average="macro")
print(f"\n  Full-dataset resubstitution metrics (not generalization estimates):")
print(f"    Accuracy:  {full_acc:.4f}")
print(f"    Macro-F1:  {full_f1:.4f}")

## 10 — Classification Report & Confusion Matrix
Per-class precision, recall, and F1-score for the best model fitted on
the full dataset, plus a normalized confusion matrix heatmap.

In [ ]:
print(f"{'='*70}")
print(f" CLASSIFICATION REPORT — {best_model_name} (Full Dataset)")
print(f"{'='*70}\n")
print(classification_report(y_actual_labels, y_pred_labels, zero_division=0))

# ── Normalized Confusion Matrix ──
cm = confusion_matrix(y_actual_labels, y_pred_labels, labels=le.classes_)
cm_pct = cm.astype("float") / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor(DARK)

sns.heatmap(cm_pct, annot=True, fmt=".1%", cmap="YlOrRd",
            xticklabels=le.classes_, yticklabels=le.classes_,
            ax=ax, cbar_kws={"shrink": 0.8},
            linewidths=0.5, linecolor="#2A2D35")

ax.set_xlabel("Predicted Genre", fontsize=11)
ax.set_ylabel("Actual Genre", fontsize=11)
ax.set_title(f"Normalized Confusion Matrix — {best_model_name}",
             fontsize=13, pad=12, color="white")
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig("fig_confusion_matrix.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

## 11 — Export `predictions.csv`
Final deliverable containing original indices, album metadata, actual genres,
and predicted genres.

In [ ]:
predictions_df = pd.DataFrame({
    "original_index": original_indices,
    "release_name": df.loc[mask_top10, "release_name"].values,
    "artist_name": df.loc[mask_top10, "artist_name"].values,
    "actual_genre": y_actual_labels,
    "predicted_genre": y_pred_labels
})

predictions_df.to_csv("predictions.csv", index=False)

print(f"Predictions exported: predictions.csv")
print(f"  Shape: {predictions_df.shape}")
predictions_df.head(10)

## 12 — Pipeline Execution Summary

In [ ]:
print(f"""\n{'='*70}\n PIPELINE EXECUTION COMPLETE\n{'='*70}\n""")
print(f" Feature Engineering:")
print(f"   • TF-IDF vectorization with positional decay (α=0.50)")
print(f"   • Truncated SVD: {M.shape[1]} tags → {N_COMPONENTS} components")
print(f"   • Combined matrix: {N_COMPONENTS} SVD + 3 numerical = {X_filtered.shape[1]} features")
print(f"")
print(f" Target Variable:")
print(f"   • Dominant genre = first genre in primary_genres")
print(f"   • Top 10 genres: {len(le.classes_)} classes, {X_filtered.shape[0]} samples")
print(f"")
print(f" Cross-Validation ({N_FOLDS}-Fold Stratified):")
for name in models:
    a = np.mean(results[name]['accuracy'])
    f = np.mean(results[name]['macro_f1'])
    marker = ' ★' if name == best_model_name else ''
    print(f"   • {name:<25s} → Acc: {a:.4f}  |  F1: {f:.4f}{marker}")
print(f"")
print(f" Best Model: {best_model_name}")
print(f"   • CV Mean Accuracy:  {best_acc:.4f}")
print(f"   • CV Mean Macro-F1:  {best_f1:.4f}")
print(f"")
print(f" Deliverables:")
print(f"   ✓ predictions.csv ({predictions_df.shape[0]} rows)")
print(f"   ✓ fig_class_distribution.png")
print(f"   ✓ fig_cv_results.png")
print(f"   ✓ fig_confusion_matrix.png")
print(f"{'='*70}")